# Banking77 LoRA Fine-Tuning — Colab Orchestration
Clones the repo, installs deps (leaving Colab's preinstalled torch alone per outside-voice E4), and runs the pipeline: data prep -> baseline eval -> LoRA training (3 configs) -> fine-tuned eval -> report generation.

Checkpoints save to Google Drive after each epoch. Predictions persist incrementally to JSONL so a disconnect doesn't lose an eval pass (E2).

In [ ]:
# Step 0 (runs in parallel with step 1, per D10): create/verify the GitHub remote and clone (D5).
!git clone https://github.com/V1cky16112003/banking77-lora-finetuning.git repo
%cd repo

In [ ]:
# Install deps EXCEPT torch (Colab's preinstalled CUDA-matched torch build stays as-is, per E4).
!pip install -r <(grep -v '^torch' requirements.txt)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_ROOT = '/content/drive/MyDrive/banking77-lora-checkpoints'

## Step 1: Load config and data (design-doc step 1-2)

In [ ]:
from src.config import load_task_config
from src.data import load_banking77_splits

config = load_task_config()
splits = load_banking77_splits(config)

# Persist the verified label list + dataset revision back into configs/task.yaml
# (never hand-transcribe -- verify against the loaded dataset, per Approach A's note).
print(f'{len(splits.labels)} labels, dataset revision: {splits.dataset_revision}')
print(f'train={len(splits.train)} val={len(splits.val)} test={len(splits.test)} (frozen, untouched until final eval)')

## Step 2: Baseline evaluation (zero-shot + few-shot) on the frozen test set (design-doc step 3)

In [ ]:
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.evaluate import evaluate_dataset
from src.train import MODEL_NAME
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).cuda()

test_examples = [{'text': ex['text'], 'label': splits.labels[ex['label']]} for ex in splits.test]
baseline_predictions_path = Path('eval_runs/baseline_test_predictions.jsonl')
baseline_predictions_path.parent.mkdir(parents=True, exist_ok=True)
baseline_results = evaluate_dataset(base_model, tokenizer, test_examples, config, baseline_predictions_path)

## Step 3: LoRA fine-tuning -- 3 candidate configs, compared on validation (design-doc step 4)

In [ ]:
from src.train import CANDIDATE_CONFIGS, train_lora

checkpoint_paths = {}
for run_config in CANDIDATE_CONFIGS:
    output_dir = Path(CHECKPOINT_ROOT) / run_config.name
    checkpoint_paths[run_config.name] = train_lora(config, splits, run_config, output_dir)

## Step 4: Evaluate each config on validation, pick the best, confirm once on the frozen test set (design-doc step 5)

In [ ]:
# For each candidate config: load adapter, evaluate on splits.val (NOT the frozen test set),
# compute macro-F1 via src.evaluate_core.compute_macro_f1, pick the best.
# Then, and only then, run the winning config once on splits.test (frozen) for the final numbers.
# (Left as an exercise wired to evaluate_dataset() + PeftModel.from_pretrained(base_model, checkpoint_paths[name])
#  -- the pattern is identical to the baseline eval cell above, swapping the model.)

## Step 5: Generate the report (results table, per-class F1, confusion heatmap, bootstrap CI) -- item 2/3, E1

In [ ]:
!python scripts/generate_report.py eval_runs/finetuned_test_predictions.jsonl --output-dir eval_runs/report

## Step 6: Download the adapter checkpoint locally for serving (D5)
Download the winning config's checkpoint directory from Drive to your local machine before running `uvicorn src.serve:app`.